# Radiology RVU Forecasting (3-Week Moving Average)

This Databricks notebook predicts hourly RVU using a 3-week moving average for each **(Modified_Clario_Site_ID, Priority)** combination.

**Source table**: `edw_dev.matrix_lateetud.exam_data_lateetud_deduped`


## 1) Runtime Parameters
Use widgets to set the prediction date and output CSV path.


In [ ]:
# Databricks widgets
try:
    dbutils.widgets.removeAll()
except Exception:
    pass

dbutils.widgets.text("prediction_date", "2026-03-22", "Prediction Date (YYYY-MM-DD)")
dbutils.widgets.text("output_path", "dbfs:/FileStore/rvu_predictions/rvu_forecast.csv", "Output CSV Path")

prediction_date = dbutils.widgets.get("prediction_date").strip()
output_path = dbutils.widgets.get("output_path").strip()

print(f"prediction_date = {prediction_date}")
print(f"output_path     = {output_path}")


## 2) Imports and Configuration


In [ ]:
import logging
import os
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Tuple

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("Radiology3WeekForecast")


## 3) Forecasting Class
Implements data loading, validation, 3-week lookup-based forecasting, metrics, and CSV export.


In [ ]:
@dataclass
class ForecastSummary:
    prediction_date: str
    training_start: str
    training_end: str
    test_start: str
    test_end: str
    training_rows: int
    test_rows: int
    output_rows: int
    coverage_any_history_pct: float
    coverage_full_history_pct: float
    mae: float
    rmse: float
    mape_pct: float


class Radiology3WeekForecast:
    """3-week moving-average forecaster for radiology RVU demand in Databricks."""

    SOURCE_TABLE = "edw_dev.matrix_lateetud.exam_data_lateetud_deduped"
    SITE_FILTER = (1297, 1298, 1299, 1300, 1301, 1302, 1303, 1304)
    REQUIRED_COLUMNS = ["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS", "RVU"]

    def __init__(self, prediction_date: str, output_path: str):
        self.prediction_date = prediction_date
        self.output_path = output_path
        self.prediction_start = self._parse_date(prediction_date)
        self.prediction_end = self.prediction_start + timedelta(days=1)
        self.training_start = self.prediction_start - timedelta(days=21)
        self.training_end = self.prediction_start
        self.train_pdf = None
        self.test_pdf = None
        self.results_pdf = None
        self.summary = None

    @staticmethod
    def _parse_date(date_text: str) -> datetime:
        try:
            return datetime.strptime(date_text, "%Y-%m-%d")
        except ValueError as e:
            raise ValueError("prediction_date must be in YYYY-MM-DD format") from e

    @staticmethod
    def _to_local_dbfs_path(path: str) -> str:
        if path.startswith("dbfs:/"):
            return "/dbfs/" + path[len("dbfs:/"):].lstrip("/")
        return path

    def _validate_table_schema(self):
        logger.info("Validating source table schema...")
        actual_cols = set(spark.table(self.SOURCE_TABLE).columns)
        missing = [c for c in self.REQUIRED_COLUMNS if c not in actual_cols]
        if missing:
            raise ValueError(f"Missing required columns in {self.SOURCE_TABLE}: {missing}")

    def load_data(self):
        logger.info("Loading training/test data from Spark SQL...")
        self._validate_table_schema()

        site_csv = ",".join(str(s) for s in self.SITE_FILTER)

        train_sql = f"""
        SELECT
            Modified_Clario_Site_ID,
            Priority,
            date_trunc('hour', Modified_Unread_DTS) AS Modified_Unread_DTS,
            SUM(COALESCE(RVU, 0.0)) AS RVU
        FROM {self.SOURCE_TABLE}
        WHERE Modified_Clario_Site_ID IN ({site_csv})
          AND Modified_Unread_DTS >= TIMESTAMP('{self.training_start:%Y-%m-%d %H:%M:%S}')
          AND Modified_Unread_DTS <  TIMESTAMP('{self.training_end:%Y-%m-%d %H:%M:%S}')
        GROUP BY 1,2,3
        """

        test_sql = f"""
        SELECT
            Modified_Clario_Site_ID,
            Priority,
            date_trunc('hour', Modified_Unread_DTS) AS Modified_Unread_DTS,
            SUM(COALESCE(RVU, 0.0)) AS RVU
        FROM {self.SOURCE_TABLE}
        WHERE Modified_Clario_Site_ID IN ({site_csv})
          AND Modified_Unread_DTS >= TIMESTAMP('{self.prediction_start:%Y-%m-%d %H:%M:%S}')
          AND Modified_Unread_DTS <  TIMESTAMP('{self.prediction_end:%Y-%m-%d %H:%M:%S}')
        GROUP BY 1,2,3
        """

        self.train_pdf = spark.sql(train_sql).toPandas()
        self.test_pdf = spark.sql(test_sql).toPandas()

        for frame_name, pdf in [("train", self.train_pdf), ("test", self.test_pdf)]:
            if not pdf.empty:
                pdf["Modified_Unread_DTS"] = pd.to_datetime(pdf["Modified_Unread_DTS"])
                pdf["Priority"] = pdf["Priority"].astype(str)
            logger.info("%s rows: %s", frame_name, len(pdf))

        if self.train_pdf.empty:
            raise ValueError(
                "No training data found in the 21-day window. "
                f"Window: {self.training_start} to {self.training_end}"
            )

        available_sites = sorted(self.train_pdf["Modified_Clario_Site_ID"].dropna().unique().tolist())
        missing_sites = sorted(set(self.SITE_FILTER) - set(available_sites))
        if missing_sites:
            logger.warning("Sites with no training data: %s", missing_sites)

        logger.info("Training window: %s -> %s", self.training_start, self.training_end)
        logger.info("Prediction day : %s", self.prediction_start.date())

    def compute_forecast(self):
        logger.info("Computing 3-week moving-average forecast...")

        train = self.train_pdf.copy()
        test = self.test_pdf.copy()

        combos = pd.concat([
            train[["Modified_Clario_Site_ID", "Priority"]],
            test[["Modified_Clario_Site_ID", "Priority"]],
        ], ignore_index=True).drop_duplicates().reset_index(drop=True)

        if combos.empty:
            raise ValueError("No (site, priority) combinations found for forecasting.")

        hours = pd.date_range(self.prediction_start, self.prediction_end - timedelta(hours=1), freq="h")
        base = combos.assign(_k=1).merge(pd.DataFrame({"Modified_Unread_DTS": hours, "_k": 1}), on="_k").drop(columns=["_k"])

        test_actual = (
            test.rename(columns={"RVU": "RVU_actual"})
            [["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS", "RVU_actual"]]
        )
        base = base.merge(test_actual, on=["Modified_Clario_Site_ID", "Priority", "Modified_Unread_DTS"], how="left")

        lookup: Dict[Tuple[int, str, pd.Timestamp], float] = {
            (int(r.Modified_Clario_Site_ID), str(r.Priority), pd.Timestamp(r.Modified_Unread_DTS)): float(r.RVU)
            for r in train.itertuples(index=False)
        }

        lag_values = []
        for row in base.itertuples(index=False):
            site = int(row.Modified_Clario_Site_ID)
            priority = str(row.Priority)
            ts = pd.Timestamp(row.Modified_Unread_DTS)
            values = [
                lookup.get((site, priority, ts - pd.Timedelta(days=7)), np.nan),
                lookup.get((site, priority, ts - pd.Timedelta(days=14)), np.nan),
                lookup.get((site, priority, ts - pd.Timedelta(days=21)), np.nan),
            ]
            lag_values.append(values)

        lag_df = pd.DataFrame(lag_values, columns=["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"])
        out = pd.concat([base.reset_index(drop=True), lag_df], axis=1)
        out["history_points"] = out[["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"]].notna().sum(axis=1)
        out["RVU_predicted"] = out[["rvu_lag_7d", "rvu_lag_14d", "rvu_lag_21d"]].mean(axis=1)

        out = out.sort_values(["Modified_Unread_DTS", "Modified_Clario_Site_ID", "Priority"]).reset_index(drop=True)
        self.results_pdf = out

    def evaluate(self):
        logger.info("Calculating validation and summary metrics...")
        df = self.results_pdf.copy()

        coverage_any = float((df["history_points"] >= 1).mean() * 100.0) if len(df) else 0.0
        coverage_full = float((df["history_points"] == 3).mean() * 100.0) if len(df) else 0.0

        scored = df[df["RVU_actual"].notna() & df["RVU_predicted"].notna()].copy()
        if scored.empty:
            mae = rmse = mape = float("nan")
        else:
            err = scored["RVU_actual"] - scored["RVU_predicted"]
            mae = float(np.mean(np.abs(err)))
            rmse = float(np.sqrt(np.mean(np.square(err))))
            denom = scored["RVU_actual"].replace(0, np.nan)
            mape = float((np.abs(err / denom)).mean() * 100.0)

        self.summary = ForecastSummary(
            prediction_date=self.prediction_date,
            training_start=f"{self.training_start:%Y-%m-%d %H:%M:%S}",
            training_end=f"{self.training_end:%Y-%m-%d %H:%M:%S}",
            test_start=f"{self.prediction_start:%Y-%m-%d %H:%M:%S}",
            test_end=f"{self.prediction_end:%Y-%m-%d %H:%M:%S}",
            training_rows=int(len(self.train_pdf)),
            test_rows=int(len(self.test_pdf)),
            output_rows=int(len(df)),
            coverage_any_history_pct=coverage_any,
            coverage_full_history_pct=coverage_full,
            mae=mae,
            rmse=rmse,
            mape_pct=mape,
        )

        summary_df = pd.DataFrame([self.summary.__dict__])
        display(summary_df)

    def save_results(self):
        logger.info("Saving CSV output...")
        final_df = self.results_pdf[[
            "Modified_Clario_Site_ID",
            "Priority",
            "Modified_Unread_DTS",
            "RVU_actual",
            "RVU_predicted",
        ]].copy()

        local_path = self._to_local_dbfs_path(self.output_path)
        parent = os.path.dirname(local_path)

        if self.output_path.startswith("dbfs:/"):
            dbutils.fs.mkdirs(self.output_path.rsplit("/", 1)[0])
        if parent:
            os.makedirs(parent, exist_ok=True)

        final_df.to_csv(local_path, index=False)
        logger.info("Saved results to %s", self.output_path)
        display(final_df.head(20))

    def run(self):
        logger.info("Starting forecast pipeline...")
        self.load_data()
        self.compute_forecast()
        self.evaluate()
        self.save_results()
        logger.info("Forecast pipeline complete.")


## 4) Execute Forecast


In [ ]:
forecaster = Radiology3WeekForecast(prediction_date=prediction_date, output_path=output_path)
forecaster.run()


## 5) Optional Visualization
Actual vs predicted hourly RVU for combinations where actuals are available.


In [ ]:
import matplotlib.pyplot as plt

viz_df = forecaster.results_pdf.copy()
viz_df = viz_df[viz_df["RVU_actual"].notna()].copy()

if viz_df.empty:
    print("No actual values available on prediction day for charting.")
else:
    sample_combo = viz_df[["Modified_Clario_Site_ID", "Priority"]].drop_duplicates().iloc[0]
    site = int(sample_combo["Modified_Clario_Site_ID"])
    priority = str(sample_combo["Priority"])
    plot_df = viz_df[(viz_df["Modified_Clario_Site_ID"] == site) & (viz_df["Priority"] == priority)].copy()

    plt.figure(figsize=(12, 4))
    plt.plot(plot_df["Modified_Unread_DTS"], plot_df["RVU_actual"], marker="o", label="Actual")
    plt.plot(plot_df["Modified_Unread_DTS"], plot_df["RVU_predicted"], marker="o", label="Predicted")
    plt.title(f"Hourly RVU | Site {site} | Priority {priority}")
    plt.xlabel("Hour")
    plt.ylabel("RVU")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
